# Tutorial 9 — Lumpy investment and inaction

**Source:** `../PS5/Problem_Set_5_Q3.py`

**In class:** PS5 Question 3; the class grid and panel are smaller than in the full source script.

Run cells from top to bottom. At each pause, predict the result before running the next cell. Plots appear below their cells; the source scripts remain the full resolution reference.


## 1. The plant's choice

At $(k,z)$ the plant compares doing nothing with paying a fixed cost and choosing $k'$:

$$V(k,z)=\max\{V_n(k,z),V_a(k,z)\},\qquad i=k'-(1-\delta)k.$$

If it does nothing, $k'=(1-\delta)k$. With investment, buying costs $i$; selling recovers only $p_s|i|$. Profit is $\pi(k,z)=zk^\nu$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm


In [ ]:
BETA, DELTA, NU = .96, .10, .60
RHO, SIGMA = .70, .30
F_COST, P_SELL = .05, .95
N_K, N_Z = 100, 7  # class setting; source uses 250, 11


## 2. Give productivity a finite Markov chain

Each row of $P$ is a probability distribution for next period's productivity. Persistent $z$ matters because today's decision depends on tomorrow's likely profit.


In [ ]:
def tauchen(rho, sigma, n, m=3.0):
    sd = sigma / np.sqrt(1 - rho**2)
    z = np.linspace(-m * sd, m * sd, n)
    w = z[1] - z[0]
    P = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if j == 0:
                P[i, j] = norm.cdf((z[0] - rho * z[i] + w / 2) / sigma)
            elif j == n - 1:
                P[i, j] = 1 - norm.cdf((z[-1] - rho * z[i] - w / 2) / sigma)
            else:
                P[i, j] = norm.cdf((z[j] - rho * z[i] + w / 2) / sigma) - norm.cdf(
                    (z[j] - rho * z[i] - w / 2) / sigma
                )
    return np.exp(z), P


In [ ]:
Z, P = tauchen(RHO, SIGMA, N_Z)
print('Productivity nodes:', np.round(Z, 3))
print('Transition row sums:', P.sum(axis=1))


## 3. Build the capital grid and movement cost

The frictionless target solves $\nu zk^{\nu-1}=r+\delta$. The grid extends below and above all targets. `COST[i,j]` is the cost of moving from current grid point $k_i$ to candidate $k'_j$.


In [ ]:
R_RATE = 1 / BETA - 1
KSTAR = (NU * Z / (R_RATE + DELTA)) ** (1 / (1 - NU))
K = np.exp(np.linspace(np.log(KSTAR.min() * 0.25), np.log(KSTAR.max() * 1.6), N_K))
PI = Z[None, :] * K[:, None] ** NU

# cost of moving from k to k', for every (k, k') pair
_KOLD = (1 - DELTA) * K[:, None]
_GAP = K[None, :] - _KOLD
COST = np.where(_GAP >= 0, _GAP, P_SELL * _GAP)


In [ ]:
print('Capital grid:', round(K[0], 3), 'to', round(K[-1], 3))
print('Cost matrix shape:', COST.shape)
print('Buying example:', COST[20, 80], '; selling example:', COST[80, 20])


## 4. Compare two actions at one state

Before iterating, take a crude guess $V=\pi/(1-\beta)$. The adjust option searches all candidate $k'$. The no-adjust option interpolates continuation value at $(1-\delta)k$.


In [ ]:
V_guess = PI/(1-BETA)
EV = V_guess @ P.T
ik, iz = N_K//2, N_Z//2
adjust_values = (1-F_COST)*PI[ik,iz] - COST[ik,:] + BETA*EV[:,iz]
no_adjust_value = PI[ik,iz] + BETA*np.interp((1-DELTA)*K[ik], K, EV[:,iz])
print('Adjust value:', round(adjust_values.max(), 3), 'No-adjust value:', round(no_adjust_value, 3))
print('Best adjust target:', round(K[adjust_values.argmax()], 3))


## 5. Iterate and find the inaction region

The full update repeats this comparison for every $(k,z)$. A fixed cost creates a band in which the plant keeps depreciating capital instead of paying to reset it.


In [ ]:
def solve(f=F_COST, cost=COST, tol=1e-8, maxit=3000):
    """Value function iteration with a discrete adjust / do-nothing choice."""
    V = PI / (1 - BETA)
    for it in range(maxit):
        EV = V @ P.T
        Va = np.empty_like(V)
        idx = np.empty(V.shape, dtype=int)
        for iz in range(N_Z):
            M = -cost + BETA * EV[None, :, iz]        # (k, k') payoff net of profit
            idx[:, iz] = M.argmax(axis=1)
            Va[:, iz] = PI[:, iz] * (1 - f) + M[np.arange(N_K), idx[:, iz]]
        Vn = np.empty_like(V)
        for iz in range(N_Z):
            Vn[:, iz] = PI[:, iz] + BETA * np.interp((1 - DELTA) * K, K, EV[:, iz])
        Vnew = np.maximum(Va, Vn)
        gap = np.max(np.abs(Vnew - V))
        V = Vnew
        if gap < tol:
            break
    adjust = Va > Vn
    kpol = np.where(adjust, K[idx], (1 - DELTA) * K[:, None])
    return V, adjust, kpol, it + 1, gap


In [ ]:
V, adjust, kpol, iters, gap = solve()
mid = N_Z//2
band = np.where(~adjust[:,mid])[0]
print(f'{iters} iterations, gap {gap:.2e}')
print('Median-productivity inaction band:', np.round(K[[band[0], band[-1]]], 3) if len(band) else 'none')


## 6. See the discrete decision

The policy jumps at adjustment thresholds. Plot a few productivity levels and a map of states in which adjustment is chosen.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for j, label in ((1, 'low z'), (mid, 'median z'), (N_Z-2, 'high z')):
    axes[0].plot(K, kpol[:,j], label=label)
axes[0].plot(K, K, 'k--', lw=.8)
axes[0].set(xscale='log', yscale='log', xlabel='$k$', ylabel="$k'$")
axes[0].legend(frameon=False)
axes[1].imshow(adjust.T, origin='lower', aspect='auto', cmap='Greys',
               extent=[np.log(K[0]), np.log(K[-1]), np.log(Z[0]), np.log(Z[-1])])
axes[1].set(xlabel='$\\log k$', ylabel='$\\log z$', title='Dark: adjust')
plt.tight_layout(); plt.show()


## 7. Simulate many plants

For each plant and year, investment rate is $i/k=[k'-(1-\delta)k]/k$. The panel discards initial years so the distribution is not driven by the arbitrary starting state. The smaller class panel is quick; the source uses 6,000 plants and 400 years.


In [ ]:
def simulate(kpol, T=400, N=6000, burn=100, seed=1):
    """Panel of firms; returns the matrix of investment rates i/k."""
    rng = np.random.default_rng(seed)
    cum = P.cumsum(axis=1)
    iz = rng.integers(0, N_Z, N)
    k = np.full(N, K[N_K // 2], dtype=float)
    out = np.empty((T - burn, N))
    for t in range(T):
        kn = np.empty(N)
        for j in range(N_Z):                          # one interp call per z state
            m = iz == j
            if m.any():
                kn[m] = np.interp(k[m], K, kpol[:, j])
        rate = (kn - (1 - DELTA) * k) / k
        if t >= burn:
            out[t - burn] = rate
        k = kn
        iz = (rng.random(N)[:, None] > cum[iz]).sum(axis=1)
    return out


def facts(R):
    r = R.ravel()
    return dict(
        mean=r.mean() * 100,
        spike=(r > 0.20).mean() * 100,
        inaction=(np.abs(r) < 0.01).mean() * 100,
        negative=(r < 0).mean() * 100,
        serial=np.corrcoef(R[:-1].ravel(), R[1:].ravel())[0, 1],
    )


In [ ]:
R = simulate(kpol, T=180, N=1000, burn=40, seed=1)
print('Panel shape (years, plants):', R.shape)
print('First ten rates:', np.round(R[0,:10], 3))


## 8. Compare moments with the data

Look at mean investment, spikes ($i/k>20\%$), near-zero investment ($|i/k|<1\%$), negative investment, and serial correlation. The model is intended to match some facts and visibly miss inaction.


In [ ]:
model = facts(R)
data = dict(mean=12.2, spike=18.0, inaction=8.0, negative=10.0, serial=.007)
for key in data:
    print(f'{key:>8}: model {model[key]:7.2f}; data {data[key]:7.2f}')


## 9. Which friction changes which moment?

Raise the fixed cost $f$ and lower the resale price $p_s$ separately. The former should widen the inaction band; the latter should make negative investment rarer. These extra solves can run after the main discussion.


In [ ]:
for fixed, resale in ((.01,.95), (.05,.95), (.10,.95), (.05,.80), (.05,1.00)):
    cost = np.where(_GAP>=0, _GAP, resale*_GAP)
    _, _, kp, _, _ = solve(f=fixed, cost=cost)
    moments = facts(simulate(kp, T=180, N=1000, burn=40, seed=1))
    print(f'f={fixed:.2f}, ps={resale:.2f}: inaction={moments["inaction"]:.1f}%, negative={moments["negative"]:.1f}%')


## 10. Remove both nonconvexities

Set $f=0$ and $p_s=1$. The plant can reset toward its target without a discrete penalty. Discuss why the baseline model's inaction rate is much higher than the data's: real plants also make small replacement investments.


In [ ]:
frictionless_cost = np.where(_GAP>=0, _GAP, _GAP)
_, adj0, kp0, _, _ = solve(f=0, cost=frictionless_cost)
baseline, frictionless = facts(R), facts(simulate(kp0, T=180, N=1000, burn=40, seed=1))
print('Inaction: baseline', round(baseline['inaction'],1), '%, frictionless', round(frictionless['inaction'],1), '%')
print('Share of states with adjustment:', round(adj0.mean()*100,1), '%')
